# 49 · Classic RDD map/filter/mapPartitions

**Scenario:** Learn legacy partition APIs using only the genuine asset DataFrame and compare with native expressions.

**Source:** Databricks-provided `samples.nyctaxi.trips` only. File formats, nested structures, replays and histories in this lab are derived from that asset. No hand-created record dataset, third-party download, pandas or generated replacement data is used.

**Prerequisites:** Run notebook 00 first. Each notebook is independent; run its cells in order. Mode: **classic**. Dedicated classic compute only for the optional RDD block.

**Learning-size sample:** `max_rows` defaults to 2,000. Sorting/limiting caps result size, not the underlying scan cost. Assertions use relationships instead of hardcoded dataset counts because sample assets may change.


## Step 1 — Load and inspect the Databricks asset

Read the built-in sample and normalize column names. A stable content fingerprint helps with replay examples; it is not a guaranteed real-world trip key. The original `samples` asset is always read-only.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
dbutils.widgets.text('max_rows','2000')
MAX_ROWS=int(dbutils.widgets.get('max_rows'))
assert 100<=MAX_ROWS<=20000, 'Use 100–20000 rows for these bounded learning labs.'
SOURCE_ASSET='samples.nyctaxi.trips'
try:
    asset=(spark.read.table(SOURCE_ASSET)
        .select('tpep_pickup_datetime','tpep_dropoff_datetime','trip_distance',
                'fare_amount','pickup_zip','dropoff_zip')
        .orderBy('tpep_pickup_datetime','tpep_dropoff_datetime','pickup_zip',
                 'dropoff_zip','trip_distance','fare_amount')
        .limit(MAX_ROWS))
    assert asset.limit(1).count()==1, 'Databricks sample asset is empty.'
except Exception as exc:
    raise RuntimeError('This course requires the Databricks-provided samples.nyctaxi.trips asset. '
        'Check Unity Catalog/sample access on the attached compute. No synthetic or external fallback is used.') from exc
df=(asset.select(F.col('tpep_pickup_datetime').cast('timestamp').alias('pickup_ts'),
    F.col('tpep_dropoff_datetime').cast('timestamp').alias('dropoff_ts'),
    F.col('trip_distance').cast('double').alias('distance'),
    F.col('fare_amount').cast('double').alias('fare'),
    F.col('pickup_zip').cast('string').alias('pickup_zip'),
    F.col('dropoff_zip').cast('string').alias('dropoff_zip')))
df=df.withColumn('trip_id',F.sha2(F.to_json(F.struct(*[F.col(c) for c in df.columns]),
    options={'ignoreNullFields':'false'}),256))
df=df.withColumn('pickup_date',F.to_date('pickup_ts'))
df=df.withColumn('duration_sec',F.unix_timestamp('dropoff_ts')-F.unix_timestamp('pickup_ts'))
print('Source:',SOURCE_ASSET,'sample limit:',MAX_ROWS)
# trip_id is a content fingerprint, NOT an official source/business trip identifier.
# Identical source rows intentionally have identical fingerprints.
df.show(5,truncate=False)


## Step 2 — Keep legacy APIs opt-in

RDD APIs are unavailable on serverless and standard access mode. Use dedicated classic compute for this exercise; DataFrame APIs remain the default course path.


In [ ]:
RUN_CLASSIC_RDD=False
print('Legacy RDD exercise enabled:',RUN_CLASSIC_RDD)
native=df.filter('distance > 0').groupBy('pickup_zip').agg(F.count('*').alias('trips'))
native.show(5)


**Expected check:** Default execution stays in DataFrame APIs.


## Step 3 — Map partitions without external side effects

Each input partition produces a bounded count summary. No fabricated source records, SparkContext.parallelize or pandas conversion is used.


In [ ]:
if RUN_CLASSIC_RDD:
    def summarize_partition(rows):
        n=0
        for row in rows:
            if row['distance'] is not None and row['distance']>0: n+=1
        yield n
    partition_counts=df.select('distance').rdd.mapPartitions(summarize_partition).collect()
    assert sum(partition_counts)==df.filter('distance > 0').count()
    print('RDD partition count:',len(partition_counts),'valid rows:',sum(partition_counts))
else:
    print('RDD execution skipped; use the native DataFrame equivalent above.')


**Expected check:** On dedicated classic compute, partition totals reconcile with the DataFrame filter count.


## Practice

Compare foreachPartition and mapPartitions when you need to return rejected records and a summary.

## Solution guide

foreachPartition returns no result to the driver. mapPartitions can emit outcomes, but retry/recomputation can repeat external side effects. Use transactional/idempotent sink designs and materialize outcomes deliberately.

## Production considerations

Keep source lineage and stable business keys. Measure on representative volumes, review nulls and time semantics, and make retries idempotent. Sample-data checks demonstrate mechanics; they do not establish production performance or exactly-once external writes.
